In [15]:
import sys
sys.path.append('..')
import torch
import soundfile as sf
import json
import matplotlib.pyplot as plt
from scipy.signal import find_peaks
import numpy as np
import argparse
import random

from encoder import FMEncoder, compute_spectrogram_cqt
from fm_ddsp import fm_renderer, make_mod_matrix
from nnAudio.features import CQT2010v2
from generate_dataset import generate_dataset
from train import train

%load_ext autoreload
%autoreload 2

# Stage 1 — Single modulator, single carrier
    # Fixed f0=440, one modulator → one carrier, vary ratio of modulator and level. Encoder learns the relationship between ratio and sideband position.
# Stage 2 — Two operators, fixed algorithm
    # Still fixed f0, introduce a second modulator. Encoder learns to handle multiple simultaneous modulators.
# Stage 3 — Variable f0, fixed algorithm
    # Now vary f0 across MIDI range. Encoder learns pitch invariance — same timbre at different pitches should give same parameters.
# Stage 4 — Multiple algorithms
    # Introduce algorithm variety. Encoder learns to distinguish different routing topologies.
# Stage 5 — Full dataset
    # Everything varying simultaneously.

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [9]:
def stage1_params():
    ratio_choices = [0.5, 1.0, 1.5, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0]
    # only use operator 2
    ratio_op2 = random.choice(ratio_choices)
    mod_level_op2 = torch.rand(1)
    return {
        'f0': 440.0,
        'algorithm': 'algo_2',
        'mod_values': torch.tensor([0,0,0,0,0,0,1], dtype=torch.float32),
        'ratios': torch.tensor([1,1,ratio_op2,1], dtype = torch.float32),
        'levels': torch.tensor([0,0,mod_level_op2, 1], dtype = torch.float32),
        'carrier_weights': torch.tensor([0.0, 0.0, 0.0, 1.0], dtype = torch.float32)
    }

In [11]:
save_dir_pc = "B:\\TrainingData\\stage1"
#save_dir = '../data/stage1',
args = argparse.Namespace(
    n_examples = 40000,
    save_dir = save_dir_pc,
    Fs = 16000,
    duration = 1.0,
    seed = 127,
    overwrite=True
)
generate_dataset(args, param_fn = stage1_params)

Low pass filter created, time used = 0.0010 seconds
num_octave =  7
No early downsampling is required, downsample_factor =  1
Early downsampling filter created,                         time used = 0.0000 seconds
CQT kernels created, time used = 0.0010 seconds
Generating example 00000/40000
Generating example 04000/40000
Generating example 08000/40000
Generating example 12000/40000
Generating example 16000/40000
Generating example 20000/40000
Generating example 24000/40000
Generating example 28000/40000
Generating example 32000/40000
Generating example 36000/40000
Generating example 39999/40000


In [16]:
# train stage 1
args = argparse.Namespace(
    data_dir = save_dir_pc,
    Fs = 16000,
    duration = 1.0,
    lr = 0.0001,
    n_epochs = 10
)
train(args)

Low pass filter created, time used = 0.0000 seconds
num_octave =  7
No early downsampling is required, downsample_factor =  1
Early downsampling filter created,                         time used = 0.0000 seconds
CQT kernels created, time used = 0.0010 seconds


C:\Users\Marcus\AppData\Local\Programs\Python\Python312\Lib\site-packages\nnAudio\utils.py:510: UserWarning: 
input size = torch.Size([1, 1, 250])	kernel size = 512
padding with reflection mode might not be the best choice, try using constant padding
  warnings.warn(


Average epoch loss: 0.01279461816251278
Average weight change: 0.0015965994680300355
Average epoch loss: 0.011487619803845883
Average weight change: 0.0011312544811517
Average epoch loss: 0.010978646744415165
Average weight change: 0.001019494142383337
Average epoch loss: 0.010413214453682304
Average weight change: 0.0009648010600358248
Average epoch loss: 0.010640466982126236
Average weight change: 0.0009841466089710593
Average epoch loss: 0.009865615535154939
Average weight change: 0.0010595727944746614
Average epoch loss: 0.010194115375354886
Average weight change: 0.0013842798070982099
Average epoch loss: 0.009784631710872054
Average weight change: 0.0011183674214407802
Average epoch loss: 0.00982793505191803
Average weight change: 0.0010817034635692835
Average epoch loss: 0.00954539792239666
Average weight change: 0.0008701084298081696
